# OpenAI Prompts

In [0]:
import asyncio
import os

import aiohttp
import nest_asyncio
import openai
import pandas as pd
import numpy as np

from tqdm.notebook import tqdm

from genderize import Genderize
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
INSERT FULL PROFESSORS AND ASSISTANT PROFESSORS DATASETS CELL

In [0]:
def get_gender_genderize(first_name: str, last_name: str) -> str:
    try:
        full_name = f"{first_name} {last_name}".strip()
        genderize = Genderize(
            user_agent='GenderizeDocs/0.0',
            api_key=API_KEY,
            timeout=5.0
        )
        result = genderize.get([full_name], country_id="PL")
        gender = result[0].get('gender', 'unknown')
        probability = result[0].get('probability', 0.5)
        return f"{gender} {probability}"
    except Exception:
        return "unknown 0.5"

get_gender_genderize_udf = udf(get_gender_genderize, StringType())

In [0]:
openai.api_key = API_KEY

prompt_system = "You are a specialist in Polish academic careers. Your role is to infer scientists' binary gender with numerical floating point probability score in range from 0 to 100. Specify the gender only as 'male', 'female' or 'unknown' as first output word and numerical probability score as second word with no additional text."

def generate_prompts_prof(row):
    return [
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} from {row.INSTITUTION}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} who was born in {row.BIRTHYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} who received a full professorship title in {row.DEGREEYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} from {row.INSTITUTION} who was born in {row.BIRTHYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} from {row.INSTITUTION} who received a full professorship title in {row.DEGREEYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} who was born in {row.BIRTHYEAR} and received a full professorship title in {row.DEGREEYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} from {row.UCZELNIA} who was born in {row.BIRTHYEAR} and received a full professorship title in {row.DEGREEYEAR}?",
    ]

def generate_prompts_dr(row):
    return [
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} from {row.INSTITUTION}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} who was born in {row.BIRTHYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} who received a doctoral degree in {row.DEGREEYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} from {row.INSTITUTION} who was born in {row.BIRTHYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} from {row.INSTITUTION} who received a doctoral degree in {row.DEGREEYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} who was born in {row.BIRTHYEAR} and received a doctoral degree in {row.DEGREEYEAR}?",
        f"What is the gender of {row.FIRSTNAME} {row.LASTNAME} from {row.INSTITUTION} who was born in {row.BIRTHYEAR} and received a doctoral degree in {row.DEGREEYEAR}?",
    ]

In [0]:
semaphore = asyncio.Semaphore(100)

async def ask_openai(session: aiohttp.ClientSession, prompt: str) -> str:
    try:
        async with session.post(
            "https://api.openai.com/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {openai.api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": "gpt-4o",
                "messages": [
                    {"role": "system", "content": prompt_system},
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0,
                "top_p": 1
            },
            timeout=30
        ) as resp:
            data = await resp.json()
            return data["choices"][0]["message"]["content"]
    except Exception as e:
        return f"ERROR: {str(e)}"

async def process_row(row, session: aiohttp.ClientSession, df_set: str):
    prompts = (
        generate_prompts_prof(row) if df_set == "Full Professors"
        else generate_prompts_dr(row)
    )

    async def limited_call(prompt: str) -> str:
        async with semaphore:
            return await ask_openai(session, prompt)

    return await asyncio.gather(*(limited_call(p) for p in prompts))

async def process_dataframe(df, df_set: str):
    async with aiohttp.ClientSession() as session:
        tasks = [
            process_row(row, session, df_set)
            for _, row in df.iterrows()
        ]
        results = []
        for future in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            res = await future
            results.append(res)
        return results

In [0]:
nest_asyncio.apply()

async def run_processing(dataset: pd.DataFrame, label: str):
    results = await process_dataframe(dataset, label)
    for i in range(8):
        dataset[f"AI_Prompt_{i+1}"] = [r[i] for r in results]
    return spark.createDataFrame(dataset)

RESULTS_FULL_PROFESSORS = await run_processing(DATASET_FULL_PROFESSORS, "Full Professors")
RESULTS_ASSISTANT_PROFESSORS = await run_processing(DATASET_ASSISTANT_PROFESSORS, "Assistant Professors")

RESULTS_FULL_PROFESSORS = (RESULTS_FULL_PROFESSORS.withColumn('Genderize', get_gender_genderize_udf(col("FIRSTNAME"), col("LASTNAME"))))
RESULTS_ASSISTANT_PROFESSORS = (RESULTS_ASSISTANT_PROFESSORS.withColumn('Genderize', get_gender_genderize_udf(col("FIRSTNAME"), col("LASTNAME"))))

In [0]:
SAVE FULL PROFESSORS AND ASSISTANT PROFESSORS CHATGPT RESULTS CELL

# Classification metrics

In [0]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [0]:
INSERT FULL PROFESSORS AND ASSISTANT PROFESSORS CHATGPT RESULTS CELL

In [0]:
def compute_metrics(y_true, y_pred):
    accuracy = round(accuracy_score(y_true, y_pred), 4)
    precision = round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 4)
    recall = round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 4)

    return accuracy, precision, recall

In [0]:
def calculate_classification_metrics(dataframes, names):
    results = []

    for df, dataset_name in zip(dataframes, names):
        pred_cols = df.columns[-9:]
        n_total = len(df)

        for col in pred_cols:
            valid = df[['GENDER', col]].dropna().copy()
            n_valid = len(valid)

            if n_valid == 0:
                continue

            y_true = valid['GENDER'].str.lower()
            y_pred = valid[col].str.split().str[0].str.lower()

            try:
                metrics = compute_metrics(y_true, y_pred)

                results.append([dataset_name, col, *metrics])
            except Exception as e:
                print(f"Error processing dataset '{dataset_name}', column '{col}': {e}")

    columns = ['Dataset', 'Parameters', 'Accuracy', 'Precision', 'Recall']

    return pd.DataFrame(results, columns=columns) if results else pd.DataFrame(columns=columns)

metrics = calculate_classification_metrics([RESULTS_FULL_PROFESSORS, RESULTS_ASSISTANT_PROFESSORS], ['Full Professors', 'Assistant Professors'])

In [0]:
display(metrics)

### By Institution type

In [0]:
def calculate_classification_metrics(dataframes, names):
    results = []

    for df, dataset_name in zip(dataframes, names):
        pred_cols = df.columns[-9:]
        group_col = df.columns[6]

        for group_name, group_df in df.groupby(group_col):
            n_total = len(group_df)

            for col in pred_cols:
                valid = group_df[['GENDER', col]].dropna().copy()
                n_valid = len(valid)

                if n_valid == 0:
                    continue

                y_true = valid['GENDER'].str.lower()
                y_pred = valid[col].str.split().str[0].str.lower()

                try:
                    accuracy, precision, recall, specificity, f1, mcc = compute_metrics(y_true, y_pred)

                    results.append([group_name, dataset_name, col, *metrics])
                except Exception as e:
                    print(f"Error in group '{group_name}', dataset '{dataset_name}', column '{col}': {e}")

    columns = ['Uni Group', 'Dataset', 'Parameters', 'Accuracy', 'Precision', 'Recall']
    
    return pd.DataFrame(results, columns=columns) if results else pd.DataFrame(columns=columns)

metrics = calculate_classification_metrics([RESULTS_FULL_PROFESSORS, RESULTS_ASSISTANT_PROFESSORS], ['Full Professors', 'Assistant Professors'])

In [0]:
display(metrics)

### Percent share

In [0]:
base_col = "GENDER"
datasets = [dataset_prof, dataset_dr]
dataset_names = ["Full Professors", "Assistant Professors"]
thresholds = [0.85, 0.9, 0.95]
group_names = ["University", "Special University", "Technical University"]

variables = datasets[0].columns[-9:]

schema = StructType([
    StructField("Dataset", StringType(), True),
    StructField("Parameters", StringType(), True),
    StructField("Percent", FloatType(), True),
    StructField("Overlap", FloatType(), True),
])

metrics = spark.createDataFrame(spark.sparkContext.emptyRDD(), schema)

for threshold in thresholds:
    for idx, (df, name) in enumerate(zip(datasets, dataset_names)):
        for col_name in variables:
            filtered = (
                df.select(base_col, col_name)
                  .withColumnRenamed(base_col, "base")
                  .withColumnRenamed(col_name, "new")
                  .withColumn("new_split", split(col("new"), r"\s+"))
                  .withColumn("new_gender", lower(element_at(col("new_split"), 1)))
                  .withColumn("new_prob", element_at(col("new_split"), 2).cast(FloatType()))
                  .filter((col("new_gender") != "") & (col("new_prob").isNotNull()))
                  .filter(col("base") == col("new_gender"))
            )

            n_initial = filtered.count()
            matched = filtered.filter(col("new_prob") >= threshold)
            n_matched = matched.count()
            overlap = n_matched / n_initial if n_initial > 0 else 0.0

            new_row = spark.createDataFrame([(name, col_name, threshold, overlap)], schema)
            metrics = metrics.union(new_row)

metrics = (
    metrics.withColumn("Percent", format_number("Percent", 2))
)

In [0]:
display(metrics)

### Percent share by Institution type

In [0]:
schema = StructType([
    StructField("Uni Group", StringType(), True),
    StructField("Dataset", StringType(), True),
    StructField("Parameters", StringType(), True),
    StructField("Percent", FloatType(), True),
    StructField("Overlap", FloatType(), True),
])

metrics = spark.createDataFrame(spark.sparkContext.emptyRDD(), schema)

for threshold in procent:
    for df, dataset_name in zip(df, df_names):
        for group in groups:
            for col_name in variables:
                filtered = (
                    df.select("INSTITUTION_TYPE", base, col_name)
                      .filter(col("INSTITUTION_TYPE") == group)
                      .withColumnRenamed(base, "base")
                      .withColumnRenamed(col_name, "new")
                      .withColumn("split_new", split(col("new"), r"\s+"))
                      .withColumn("pred_gender", lower(element_at(col("split_new"), 1)))
                      .withColumn("prob", element_at(col("split_new"), 2).cast(FloatType()))
                      .filter((col("pred_gender") != "") & col("prob").isNotNull())
                      .filter(col("base") == col("pred_gender"))
                )

                n_initial = filtered.count()
                if n_initial == 0:
                    continue

                n_matched = filtered.filter(col("prob") >= threshold).count()
                overlap = n_matched / n_initial

                new_row = spark.createDataFrame([(group, dataset_name, col_name, threshold, overlap)], schema)
                metrics = metrics.union(new_row)

metrics = (
    metrics.withColumn("Percent", format_number("Percent", 2))
)

In [0]:
display(metrics)